# Orchestration & Lakeflow Jobs

Orchestration is the glue connecting pipelines into production workflows. We'll learn to create multi-task Jobs with dependencies (DAG), configure triggers (CRON, File Arrival), set up retries and alerts, pass parameters between tasks (taskValues), and monitor execution via System Tables.

| Exam Domain | Weight |
|---|---|
| Working with Lakeflow Jobs | 16% |
| Troubleshooting, Monitoring, and Optimization (run history, monitoring) | 10% |

## Introduction to Lakeflow Jobs

**Lakeflow Jobs** (formerly Workflows) is a managed orchestration service.

| Scenario | Solution |
|----------|----------|
| ETL pipeline with multiple steps | Multi-task Job |
| Daily report at fixed time | Scheduled Job |
| Reaction to new files | File Arrival Trigger |
| ML training pipeline | Job with notebook tasks |
| Run Lakeflow Pipeline | Job with Pipeline task |

| Feature | Jobs | Lakeflow Pipelines |
|---------|------|-----|
| Orchestration | General | ETL only |
| Dependencies | Manual configuration | Automatic (DAG) |
| Data Quality | Custom code | Built-in expectations |
| Flexibility | High | Opinionated |

**Best Practice**: Use Lakeflow Pipelines for ETL, Jobs for orchestrating Pipelines + other tasks.



### Task Types in Lakeflow Jobs

Overview of all available task types in Lakeflow Jobs and when to use each one.

| Task Type | Description | Use Case |
|-----------|-------------|----------|
| Notebook | Run a Databricks notebook | ETL logic, ML training |
| Pipeline | Run a Lakeflow Declarative Pipeline | Streaming/batch pipelines |
| Python Script | Run a Python file | Utility scripts |
| SQL | Run a SQL query | DDL, reporting queries |
| JAR | Run a Java/Scala JAR | Legacy Spark jobs |
| Spark Submit | Submit a Spark application | Custom Spark apps |
| If/Else Condition | Branch based on condition | Conditional workflows |
| For Each | Iterate over a list | Parameterized batch runs |

**Repair Runs:** Re-runs only **failed and downstream tasks**, skipping successful ones — saves compute and time.

**Exam Note:** Know that repair runs skip already-successful tasks. If/Else and For Each enable conditional and iterative workflows.

## Preparing Notebooks for Job

Below are 3 simple notebooks that we'll use in the demo.

**Instructions**: 
1. Create folder `/Workspace/Users/<your-email>/jobs_demo/`
2. Copy each of the following code snippets to a separate notebook



### Task 1: Validate Source

Validates row count against `min_rows` threshold. Publishes the count to downstream tasks with `dbutils.jobs.taskValues.set()`.

In [0]:
# TASK 1: Validate Source Data
# Copy this code to notebook: task_01_validate

# Parameters from Job
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("min_rows", "100")

source_table = dbutils.widgets.get("source_table")
min_rows = int(dbutils.widgets.get("min_rows"))

# Validation
df = spark.table(source_table)
row_count = df.count()

if row_count < min_rows:
    raise Exception(f"Validation FAILED: {row_count} rows < {min_rows} minimum")

# Share results with downstream tasks as TASK VALUES.
# NOTE: dbutils.notebook.exit() does NOT create task values — its string only goes to
# a dbutils.notebook.run() caller / the run output. Outside a job, set() does nothing.
dbutils.jobs.taskValues.set(key="row_count", value=row_count)
dbutils.jobs.taskValues.set(key="source_table", value=source_table)
print(f"Validation PASSED: {row_count} rows (task values: row_count, source_table)")


### Task 2: Transform Data

Reads previous task result via `taskValues`, applies transformations (duration, cost per mile). Publishes `rows_transformed` as a task value.

**Task Values — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Set value (upstream task) | `dbutils.jobs.taskValues.set(key="row_count", value=1000)` |
| Get value (downstream task) | `val = dbutils.jobs.taskValues.get(taskKey="validate", key="row_count", default=0, debugValue=0)` |
| `debugValue` / `default` | `debugValue` is returned when the notebook runs **outside a job** (interactive); `default` when the key is missing in a job run |
| Reference in task settings | `{{tasks.validate.values.row_count}}` (e.g. If/else condition, For each inputs, task parameters) |
| Not a task value | `dbutils.notebook.exit("...")` — returns a string to `dbutils.notebook.run()` / run output only |
| Supported types | `str`, `int`, `float`, `bool`, JSON-serializable `dict`/`list` |
| Max value size | 48 KB per key |

In [0]:
# TASK 2: Transform Data
# Copy this code to notebook: task_02_transform

from pyspark.sql.functions import *
from datetime import date

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("run_date", "")

source_table = dbutils.widgets.get("source_table")
run_date = dbutils.widgets.get("run_date") or str(date.today())

# Read the task value published by the upstream "validate" task.
# debugValue -> used when running this notebook interactively (outside a job)
# default    -> used in a job run if the key was not set upstream
rows_validated = dbutils.jobs.taskValues.get(
    taskKey="validate",
    key="row_count",
    default=0,
    debugValue=0
)
print(f"Rows validated upstream: {rows_validated}")

# Transformation
print(f"Transforming: {source_table}")

df = spark.table(source_table)

df_transformed = (
    df
    .withColumn("trip_duration_minutes", 
                round((col("tpep_dropoff_datetime").cast("long") - 
                       col("tpep_pickup_datetime").cast("long")) / 60, 2))
    .withColumn("cost_per_mile", 
                when(col("trip_distance") > 0, 
                     round(col("fare_amount") / col("trip_distance"), 2))
                .otherwise(0))
    .withColumn("processing_date", lit(run_date))
)

row_count = df_transformed.count()
print(f"Transformed {row_count} rows")

df_transformed.select(
    "trip_distance", "fare_amount", "trip_duration_minutes", "cost_per_mile"
).show(5)

# Publish result for downstream tasks
dbutils.jobs.taskValues.set(key="rows_transformed", value=row_count)


### Task 3: Generate Report

Aggregates metrics (total trips, revenue, avg fare/distance). Prints summary report.

In [0]:
# TASK 3: Generate Report
# Copy this code to notebook: task_03_report

from pyspark.sql.functions import *
from datetime import datetime

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")

source_table = dbutils.widgets.get("source_table")

# Aggregations
df = spark.table(source_table)

report = df.agg(
    count("*").alias("total_trips"),
    round(sum("fare_amount"), 2).alias("total_revenue"),
    round(avg("fare_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(max("fare_amount"), 2).alias("max_fare")
).collect()[0]

# Display report
print("\n" + "="*50)
print("DAILY REPORT")
print("="*50)
print(f"Avg Fare:       ${report.avg_fare:.2f}")
print(f"Avg Distance:   {report.avg_distance:.2f} miles")
print(f"Max Fare:       ${report.max_fare:.2f}")
print("="*50)
print(f"Generated at:   {datetime.now()}")
print("="*50 + "\n")

# Publish headline metrics as task values (visible in the run's task output)
dbutils.jobs.taskValues.set(key="total_trips", value=int(report.total_trips))
dbutils.jobs.taskValues.set(key="total_revenue", value=float(report.total_revenue))


### [UI DEMO] Creating Multi-task Job

**Step 1: Create new Job**
- [ ] **Jobs & Pipelines** → Create Job
- [ ] Name: `Demo_ETL_Pipeline`

<img src="../../../assets/images/93c107ca21a54aab98249cf47db0337d.png" width="800">

- [ ] Cluster Job: Create new cluster job

<img src="../../../assets/images/a967557a143a40c0ac7ed26ce469866a.png" width="800">

**Step 2: Add Task 1 (Validate)**
- [ ] Task name: `validate`
- [ ] Type: Notebook
- [ ] Path: `/Workspace/.../task_01_validate`
- [ ] Cluster: Serverless or new Job Cluster
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

<img src="../../../assets/images/214b868309344df3a6e81f3cc2a84c13.png" width="800">

**Step 3: Add Task 2 (Transform)**
- [ ] Task name: `transform`
- [ ] Depends on: `validate`
- [ ] Path: `/Workspace/.../task_02_transform`
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

**Step 4: Add Task 3 (Report)**
- [ ] Task name: `report`
- [ ] Depends on: `transform`
- [ ] Path: `/Workspace/.../task_03_report`

<img src="../../../assets/images/a3cc387f44de4247bf275bdbd38efb84.png" width="800">

**Step 5: Run Job**
- [ ] Run now
- [ ] Show: DAG visualization
- [ ] Show: Task logs and output



## [UI DEMO 2] Medallion Pipeline Job — Ready-to-Use Config

We already have 6 production-ready notebooks in `materials/medallion/` that implement a full Medallion pipeline:

| Layer | Notebook | Input | Output |
|-------|----------|-------|--------|
| **Bronze** | `bronze_customers` | CSV files | `bronze.bronze_customers` |
| **Bronze** | `bronze_orders` | JSON files | `bronze.bronze_orders` |
| **Silver** | `silver_customers` | bronze_customers | `silver.silver_customers` |
| **Silver** | `silver_orders_cleaned` | bronze_orders | `silver.silver_orders_cleaned` |
| **Gold** | `gold_customer_orders_summary` | silver_customers + silver_orders | `gold.gold_customer_orders_summary` |
| **Gold** | `gold_daily_orders` | silver_orders | `gold.gold_daily_orders` |

**DAG Structure:**
```
bronze_customers ──→ silver_customers ──────→ gold_customer_orders_summary
bronze_orders ────→ silver_orders_cleaned ─┤
                                           └→ gold_daily_orders
```

**How to deploy:** the definition below is **YAML** (bundle resource format). Paste it in the Jobs UI YAML editor, or put it in a bundle's `resources/` folder and run `databricks bundle deploy` — see the Deploy Checklist.



### DABs YAML Configuration

Below is the **Declarative Automation Bundles** (formerly Databricks Asset Bundles, DABs) YAML definition for the Medallion pipeline Job. This was exported from a working Databricks deployment.

> **Before deploying:** Update `notebook_path` and `parameters` to match your environment. No compute is declared, so the tasks run on **serverless jobs compute**.

```yaml
resources:
  jobs:
    job_medalion_run:
      name: job_medalion_run
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/bronze_customers
            source: WORKSPACE

        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/bronze_orders
            source: WORKSPACE

        - task_key: silver_customers
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/silver_customers
            source: WORKSPACE

        - task_key: silver_orders_cleaned
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/silver_orders_cleaned
            source: WORKSPACE

        - task_key: gold_customer_order_summary
          depends_on:
            - task_key: silver_customers
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/gold_customer_orders_summary
            source: WORKSPACE

        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/gold_daily_orders
            source: WORKSPACE

      queue:
        enabled: true
      parameters:
        - name: catalog
          default: <YOUR_CATALOG>
        - name: source_path
          default: /Volumes/<YOUR_CATALOG>/default/datasets
```

| Element | Description |
|---------|-------------|
| *(no compute key)* | Tasks run on **serverless jobs compute**. On classic compute use `job_cluster_key` (job cluster) — `existing_cluster_id` (all-purpose cluster) is for ad-hoc tests only |
| `depends_on` | Defines task dependencies — creates the DAG |
| `parameters` | Job-level parameters passed to all notebooks via `dbutils.widgets` |
| `queue.enabled` | Queues runs if max concurrent reached |
| `source: WORKSPACE` | Notebook is in Workspace (not Repos) |

### Deploy Checklist

**Option A — YAML editor (UI):**
1. [ ] **Jobs & Pipelines** → Create Job
2. [ ] Next to **Run now** click the kebab menu **⋮** → **Edit as YAML**
3. [ ] Paste the `tasks:` / `parameters:` content of the YAML above (replace placeholders)
4. [ ] Save → Run now

**Option B — Declarative Automation Bundle (CLI):**
```bash
# Save the YAML above as resources/medallion_job.yml inside a bundle, then:
databricks bundle validate -t dev
databricks bundle deploy   -t dev
databricks bundle run      -t dev job_medalion_run
```
> `databricks jobs create --json @job.json` also exists, but it expects **Jobs API JSON**, not this YAML.

**After deployment — show participants:**
- [ ] DAG visualization (fan-out at Bronze, fan-in at Gold)
- [ ] Task-level parameters (catalog, schema, source_path)
- [ ] Trigger: set to PAUSED — run manually for demo
- [ ] Run → observe sequential layer execution (Bronze → Silver → Gold)
- [ ] Show Repair Run: intentionally fail one task → repair reruns only failed + downstream


## [UI DEMO] Triggers and Schedule

How to configure different trigger types for Lakeflow Jobs — scheduled (CRON), file arrival, continuous, and manual triggers.

| Trigger Type | Usage | Example |
|---|---|---|
| **Scheduled** | Fixed schedule (CRON) | `0 0 2 * * ?` — daily at 2:00 |
| **File arrival** | Reaction to new files | New file in UC Volume |
| **Continuous** | Continuous processing | Streaming-like |
| **Manual** | On-demand | Testing |

**Exam Note:** Know CRON syntax and File Arrival trigger configuration.

### Trigger Configuration Checklist

Step-by-step instructor checklist for demonstrating trigger options in the Lakeflow Jobs UI.

**Trigger Options** (Triggers tab):

| Trigger Type | Usage | Example |
|--------------|-------|---------|
| **Scheduled** | Fixed schedule | Daily at 2:00 |
| **File arrival** | Reaction to new files | New file in `/landing/` |
| **Continuous** | Continuous processing | Streaming-like |
| **Manual** | On-demand | Testing |

<img src="../../../assets/images/6e23746ce063491fa8afb3dea6268a1d.png" width="800">

**Demo: Scheduled Trigger**
- [ ] Add trigger → Scheduled
- [ ] Cron expression: `0 0 2 * * ?` (daily at 2:00)
- [ ] Timezone: `Europe/Warsaw`
- [ ] Show: Preview next runs

<img src="../../../assets/images/cf3cfc85162a466eb77e15e20df5c15c.png" width="800">

**Demo: File Arrival Trigger** (optional)
- [ ] Add trigger → File arrival
- [ ] URL: Unity Catalog Volume path
- [ ] Min files: 1
### Useful CRON Expressions

Common CRON patterns for scheduling Lakeflow Jobs at various intervals.

```
0 0 2 * * ?        # Daily at 2:00
0 0 * * * ?        # Every hour
0 0 9 ? * MON-FRI  # Mon-Fri at 9:00
0 0 0 1 * ?        # First day of month
0 */15 * * * ?     # Every 15 minutes
```



## [UI DEMO] Options, Retry and Alerting

**Task-level:** Timeout, Retries (count + delay)  
**Job-level:** Max concurrent runs, Job timeout  
**Notifications:** Email (on failure/success), Webhooks (Slack/Teams via Destinations)

| Scenario | Retry? | Why |
|----------|--------|-----|
| Network timeout | Yes | Transient error |
| API rate limit | Yes | Transient error |
| Data quality issue | No | Retry won't fix data |
| Code bug | No | Retry won't fix code |



### Control Flow as Code — Retries, If/else, For each

The task-type table above in one compact job definition (bundle YAML; the Jobs API JSON uses the same field names):

```yaml
resources:
  jobs:
    control_flow_demo:
      name: control_flow_demo
      parameters:
        - name: tables
          default: '["samples.nyctaxi.trips", "samples.tpch.orders"]'
      tasks:
        - task_key: validate                      # sets task value row_count
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/jobs_demo/task_01_validate
          max_retries: 2                          # retry transient failures ...
          min_retry_interval_millis: 60000        # ... waiting >= 60 s between attempts
          retry_on_timeout: false

        - task_key: enough_rows                   # If/else condition task
          depends_on:
            - task_key: validate
          condition_task:
            op: GREATER_THAN                      # EQUAL_TO, NOT_EQUAL, GREATER_THAN(_OR_EQUAL), LESS_THAN(_OR_EQUAL)
            left: "{{tasks.validate.values.row_count}}"
            right: "100"

        - task_key: check_each_table              # For each task — runs only on the TRUE branch
          depends_on:
            - task_key: enough_rows
              outcome: "true"
          for_each_task:
            inputs: "{{job.parameters.tables}}"   # JSON array
            concurrency: 2                        # parallel iterations (default 1)
            task:
              task_key: check_one_table
              notebook_task:
                notebook_path: /Workspace/Users/<your-email>/jobs_demo/task_01_validate
                base_parameters:
                  source_table: "{{input}}"       # current element
```

> **Exam Note:** retries are **per task** (`max_retries`, `min_retry_interval_millis`); branching = `condition_task` + `depends_on.outcome: "true"/"false"`; looping = `for_each_task` with `{{input}}`.


## Serverless Compute for Jobs

Since 2025, Databricks offers **Serverless compute** as the default option for Lakeflow Jobs. Serverless eliminates cluster management — tasks start in seconds with auto-scaled, ephemeral compute.

| Aspect              | Classic Clusters                  | Serverless Compute           |
|---------------------|----------------------------------|------------------------------|
| **Startup time**    | 3-10 minutes                     | Seconds                      |
| **Cluster management** | Manual sizing (workers, instance types) | Fully automatic         |
| **Cost model**      | Per-VM (DBU + cloud infra)       | Per-DBU only (no infra cost) |
| **Spot instances**  | Configurable (cost savings)      | Not applicable               |
| **Auto-scaling**    | Configurable (min/max workers)   | Built-in, transparent        |
| **Custom libraries**| Full control (init scripts, pip) | Limited (pip only, no init scripts) |
| **GPU support**     | Yes                              | No                           |
| **Use case**        | Long-running, GPU, custom libs, cost-sensitive batch | ETL, SQL, notebooks, triggered jobs |

**How to enable:** In the Job task configuration, select **Serverless** as the compute type instead of specifying a cluster.

#### When to Use Classic Clusters Instead

| Scenario                | Why Classic?                                  |
|-------------------------|-----------------------------------------------|
| GPU / ML training       | Serverless doesn't support GPU instances      |
| Network isolation (Private Link) | Classic clusters support custom VNet/VPC |
| Custom init scripts     | Serverless only supports pip packages         |
| Specific instance types needed | Serverless auto-selects instance types |
| Long-running batch (hours) | Spot instances on classic can be significantly cheaper |

> **Pro Tip:** Serverless compute is ideal for **triggered jobs** that need fast response times (file arrival, table update). For cost-sensitive scheduled batch jobs running for hours, classic clusters with spot instances may still be 30-50% cheaper.

## Demo: Widgets and Parameters

Databricks Widgets allow you to parameterize notebooks.



In [0]:
# Widget types

# Text - any text
dbutils.widgets.text("environment", "dev", "Environment")

# Dropdown - select from list
dbutils.widgets.dropdown("region", "EU", ["EU", "US", "APAC"], "Region")

# Combobox - dropdown with typing option
dbutils.widgets.combobox("table", "orders", ["orders", "customers", "products"], "Table")

# Multiselect - multiple selection
dbutils.widgets.multiselect("columns", "id", ["id", "name", "date", "amount"], "Columns")

In [0]:
# Getting values
environment = dbutils.widgets.get("environment")
region = dbutils.widgets.get("region")
table = dbutils.widgets.get("table")
columns = dbutils.widgets.get("columns")  # returns comma-separated string

print(f"Environment: {environment}")
print(f"Region: {region}")
print(f"Table: {table}")
print(f"Columns: {columns}")

In [0]:
# Dynamic parameters in Job
# These values are available when notebook is run as a task in Job

dynamic_params = {
    "{{job.start_time.iso_date}}": "Run date (YYYY-MM-DD)",
    "{{job.start_time}}": "Full timestamp",
    "{{job.id}}": "Job ID",
    "{{run.id}}": "Current run ID",
    "{{task.name}}": "Current task name"
}

for param, description in dynamic_params.items():
    print(f"{param:35} -> {description}")

In [0]:
# Widget cleanup
dbutils.widgets.removeAll()

## Monitoring via System Tables

Key tables: `system.lakeflow.job_run_timeline` (run history: start/end time, result state, trigger type) and `system.lakeflow.jobs` (job metadata, including the job **name**).

Three things to know before you query them:
- **Account-wide:** system tables hold data from **every workspace in the account** → always filter on `workspace_id` (a STRING column).
- **Job names come from `system.lakeflow.jobs`:** `run_name` in the timeline is set for one-time (submitted) runs but is NULL for runs of saved jobs. Join on `workspace_id` + `job_id` and keep the latest row per job (`ORDER BY change_time DESC`) — the jobs table keeps one row per change.
- **One run ≠ one row:** a run longer than 1 hour is split into hourly rows, and `result_state` is set only on the final row. Aggregate per `run_id` first (`MIN` start, `MAX` end, `MAX_BY(result_state, period_end_time)` = final state) before counting runs or averaging durations.

In [0]:
from databricks.sdk import WorkspaceClient

# System tables are account-wide -> filter on THIS workspace (workspace_id is a STRING column)
ws_id = WorkspaceClient().get_workspace_id()
print(f"workspace_id = {ws_id}")

# Latest timeline rows in this workspace (a run longer than 1 h appears as several hourly rows)
spark.sql(f"""
    SELECT
        DATE(period_start_time) AS run_date,
        period_start_time       AS start_time,
        job_id,
        run_id,
        run_name,               -- NULL for runs of saved jobs (name: system.lakeflow.jobs)
        result_state,           -- set on the run's final row: SUCCEEDED, FAILED, ERROR, CANCELLED, ...
        trigger_type
    FROM system.lakeflow.job_run_timeline
    WHERE workspace_id = '{ws_id}'
    ORDER BY period_start_time DESC
    LIMIT 20
""").display()

In [0]:
# Per job, per day: runs, failed runs, average duration (last 7 days, this workspace)
# 1) runs: collapse the hourly timeline rows into ONE row per run_id
#    (MAX_BY = final result_state, so a repaired run counts once, with its final state)
# 2) jobs: latest name per job from system.lakeflow.jobs (one row per job change)
# Uses ws_id from the previous cell.
spark.sql(f"""
    WITH runs AS (
      SELECT workspace_id, job_id, run_id,
             MIN(period_start_time) AS start_time,
             MAX(period_end_time)   AS end_time,
             MAX_BY(result_state, period_end_time) AS result_state
      FROM system.lakeflow.job_run_timeline
      WHERE workspace_id = '{ws_id}' AND period_start_time >= current_date() - INTERVAL 7 DAYS
      GROUP BY ALL
    ),
    jobs AS (
      SELECT workspace_id, job_id, name,
             ROW_NUMBER() OVER (PARTITION BY workspace_id, job_id ORDER BY change_time DESC) AS rn
      FROM system.lakeflow.jobs
      WHERE workspace_id = '{ws_id}'
    )
    SELECT DATE(r.start_time) AS run_date,
           COALESCE(j.name, '(one-time run)') AS job_name,
           COUNT(*) AS runs,
           COUNT_IF(r.result_state IN ('FAILED','ERROR','TIMED_OUT')) AS failed,
           ROUND(AVG(TIMESTAMPDIFF(SECOND, r.start_time, r.end_time)) / 60, 1) AS avg_duration_min
    FROM runs r LEFT JOIN jobs j ON r.workspace_id = j.workspace_id AND r.job_id = j.job_id AND j.rn = 1
    GROUP BY ALL ORDER BY run_date DESC, runs DESC LIMIT 20
""").display()

## Summary

| Topic | Key Concept | Exam Keywords |
|---|---|---|
| **Multi-task Jobs** | DAG workflow, task dependencies | Task types, Repair Runs |
| **Triggers** | Scheduled (CRON), File arrival, Continuous | `0 0 2 * * ?` |
| **Options** | Timeout, Retry, Max concurrent runs | Transient vs permanent errors |
| **Alerting** | Email, Webhooks (Slack/Teams) | Notification destinations |
| **Parameters** | Widgets, dynamic values, taskValues | `dbutils.widgets`, `dbutils.jobs.taskValues` |
| **Monitoring** | System tables, success rate, duration | `system.lakeflow.job_run_timeline` |

> **← 07: Lakeflow Pipelines | Day 3 | 09: CI/CD & Automation →**

← [07 — Lakeflow Pipelines](../../day2/demo/07_lakeflow_pipelines.ipynb) | **[ README](../../../README.md)** | [09 — CI/CD & Automation →](09_cicd_and_automation.ipynb)